
# HMM Market Regime Detection from Daily OHLCV

This notebook builds a Gaussian Hidden Markov Model (HMM) for market-regime detection from daily OHLCV data.

**Workflow**
1. Load OHLCV data
2. Clean and validate data
3. Engineer return, volatility, volume, liquidity, and momentum features
4. Split chronologically: first 15 years for training, final 5 years for out-of-sample testing
5. Fit the scaler **only on the training data**
6. Compare HMMs with different numbers of states
7. Train the selected HMM
8. Label historical regimes
9. Infer regimes in the five-year test period
10. Analyze regime characteristics, transitions, persistence, and forward returns
11. Plot the detected regimes

> **Important:** `model.predict()` is useful for historical regime classification, but it is not a strict real-time forecast because the Viterbi sequence can use observations later in the sequence. A walk-forward filtered implementation is included later for genuine next-day regime forecasting.


## 1. Install / import packages

In [ ]:

# If needed, uncomment and run:
# !pip install pandas numpy matplotlib scikit-learn hmmlearn openpyxl

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

pd.set_option("display.max_columns", 100)


## 2. Load your daily OHLCV data

Expected columns: `Date`, `Open`, `High`, `Low`, `Close`, `Volume`.

If your column names differ, change the `rename_map` below.

In [ ]:

# ===== USER SETTINGS =====
FILE_PATH = "A.csv"       # Change to your file
DATE_COLUMN = "Date"

# If your data has different names, map them here.
rename_map = {
    # "timestamp": "Date",
    # "Adj Close": "Close",
    # "vol": "Volume",
}

# Load
df = pd.read_csv(FILE_PATH)

# Rename if necessary
df = df.rename(columns=rename_map)

# Parse date and sort
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
df = df.sort_values(DATE_COLUMN).set_index(DATE_COLUMN)

required_cols = ["Open", "High", "Low", "Close", "Volume"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Keep relevant columns and make numeric
df = df[required_cols].apply(pd.to_numeric, errors="coerce")

# Remove duplicate dates and missing OHLCV rows
df = df[~df.index.duplicated(keep="last")]
df = df.dropna(subset=required_cols)

print(f"Rows: {len(df):,}")
print(f"Start: {df.index.min().date()}")
print(f"End:   {df.index.max().date()}")
display(df.head())


## 3. Feature engineering

The initial feature set deliberately avoids using too many highly correlated variables. All features use current or historical information only.

In [ ]:

# Daily log return
df["logret_1d"] = np.log(df["Close"] / df["Close"].shift(1))

# Multi-day returns
df["ret_3d"] = df["Close"].pct_change(3)
df["ret_5d"] = df["Close"].pct_change(5)
df["ret_10d"] = df["Close"].pct_change(10)

# 20-day realized volatility
df["vol_avg_20"] = df["logret_1d"].rolling(20).std()

# Relative volume
df["rel_volume"] = df["Volume"] / df["Volume"].rolling(20).mean()

# Dollar volume
df["dollar_volume"] = df["Close"] * df["Volume"]

# On-balance volume
price_direction = np.sign(df["Close"].diff()).fillna(0)
df["obv"] = (price_direction * df["Volume"]).cumsum()

# Amihud illiquidity proxy
# Average |return| / dollar volume over 20 days
df["amihud"] = (
    df["logret_1d"].abs() / df["dollar_volume"].replace(0, np.nan)
).rolling(20).mean()

# Rolling skewness / kurtosis
df["skew_20"] = df["logret_1d"].rolling(20).skew()
df["kurt_20"] = df["logret_1d"].rolling(20).kurt()

# 60-day return z-score
rolling_mean_60 = df["logret_1d"].rolling(60).mean()
rolling_std_60 = df["logret_1d"].rolling(60).std()
df["zscore_ret_60"] = (
    (df["logret_1d"] - rolling_mean_60) / rolling_std_60.replace(0, np.nan)
)

# Flag days with zero return
df["zero_ret_flag"] = (df["logret_1d"] == 0).astype(int)

# Replace infinities generated by ratios
df = df.replace([np.inf, -np.inf], np.nan)

print("Feature columns:")
display(pd.DataFrame({"Feature": [
    "logret_1d", "ret_3d", "ret_5d", "ret_10d",
    "vol_avg_20", "rel_volume", "dollar_volume",
    "obv", "amihud", "skew_20", "kurt_20",
    "zscore_ret_60", "zero_ret_flag"
]}))


## 4. Select HMM features

Start with a compact set. You can experiment with additional features later.

In [ ]:

FEATURES = [
    "logret_1d",
    "ret_5d",
    "vol_avg_20",
    "rel_volume",
    "amihud",
    "zscore_ret_60",
]

model_df = df.dropna(subset=FEATURES).copy()

print(f"Rows after feature construction: {len(model_df):,}")
display(model_df[FEATURES].describe().T)


## 5. Chronological 15-year / 5-year split

This uses the **last 5 years as the test period** and everything before that as training.

In [ ]:

# Last five calendar years are held out.
test_start = model_df.index.max() - pd.DateOffset(years=5)

train = model_df[model_df.index < test_start].copy()
test = model_df[model_df.index >= test_start].copy()

print("TRAIN")
print(train.index.min().date(), "to", train.index.max().date(), f"({len(train):,} rows)")

print("\nTEST")
print(test.index.min().date(), "to", test.index.max().date(), f"({len(test):,} rows)")


## 6. Scale using training data only

This prevents future information from leaking into the model through the scaler.

In [ ]:

scaler = StandardScaler()

X_train = scaler.fit_transform(train[FEATURES])
X_test = scaler.transform(test[FEATURES])

print("Training matrix:", X_train.shape)
print("Test matrix:    ", X_test.shape)


## 7. Compare HMMs with different numbers of states

We test 2–6 regimes. Do not choose solely by likelihood; economic interpretability, persistence, and out-of-sample behavior also matter.

In [ ]:

results = []

for n_states in range(2, 7):
    # Multiple seeds reduce dependence on one random initialization.
    seed_scores = []

    for seed in [7, 21, 42, 99, 123]:
        model = GaussianHMM(
            n_components=n_states,
            covariance_type="full",
            n_iter=1000,
            tol=1e-4,
            random_state=seed
        )

        model.fit(X_train)
        seed_scores.append(model.score(X_train))

    best_score = max(seed_scores)

    # Number of free parameters is approximated for AIC/BIC.
    n_features = X_train.shape[1]
    k = (
        n_states * n_features +                         # means
        n_states * n_features * (n_features + 1) / 2 + # covariances
        n_states * (n_states - 1) +                     # transition matrix
        (n_states - 1)                                  # initial probabilities
    )

    aic = -2 * best_score + 2 * k
    bic = -2 * best_score + k * np.log(len(X_train))

    results.append({
        "states": n_states,
        "best_log_likelihood": best_score,
        "AIC": aic,
        "BIC": bic
    })

model_selection = pd.DataFrame(results)
display(model_selection)


In [ ]:

plt.figure(figsize=(9, 5))
plt.plot(model_selection["states"], model_selection["BIC"], marker="o")
plt.xlabel("Number of HMM states")
plt.ylabel("BIC")
plt.title("HMM Model Selection — BIC")
plt.xticks(model_selection["states"])
plt.grid(True, alpha=0.3)
plt.show()


## 8. Train the final HMM

Set `N_STATES` after inspecting BIC and the economic behavior of the regimes. Four is a reasonable starting point for market-regime work.

In [ ]:

N_STATES = 4
RANDOM_STATE = 42

hmm = GaussianHMM(
    n_components=N_STATES,
    covariance_type="full",
    n_iter=2000,
    tol=1e-4,
    random_state=RANDOM_STATE
)

hmm.fit(X_train)

print("Converged:", hmm.monitor_.converged)
print("Iterations:", hmm.monitor_.iter)
print("Training log-likelihood:", hmm.score(X_train))


## 9. Inspect the transition matrix

The transition matrix tells you how persistent each regime is and how likely one regime is to move into another.

In [ ]:

transition_matrix = pd.DataFrame(
    hmm.transmat_,
    index=[f"State {i}" for i in range(N_STATES)],
    columns=[f"State {i}" for i in range(N_STATES)]
)

display(transition_matrix.round(3))

# Approximate expected duration in days for each state:
# Expected duration = 1 / (1 - P(stay in state))
expected_duration = pd.Series(
    1 / (1 - np.diag(hmm.transmat_)),
    index=[f"State {i}" for i in range(N_STATES)],
    name="Expected duration (days)"
)

display(expected_duration.round(1))


## 10. Classify regimes in the training period

In [ ]:

train["regime"] = hmm.predict(X_train)

# Posterior state probabilities for historical classification
train_posteriors = hmm.predict_proba(X_train)

for i in range(N_STATES):
    train[f"state_{i}_prob"] = train_posteriors[:, i]

display(train[["Close", "regime"]].head())


## 11. Describe each regime economically

The state numbers are arbitrary. Use these statistics to decide which state looks like calm/bull, bear, crisis/high-volatility, etc.

In [ ]:

regime_stats = train.groupby("regime").agg(
    observations=("Close", "size"),
    avg_daily_return=("logret_1d", "mean"),
    daily_volatility=("logret_1d", "std"),
    avg_5d_return=("ret_5d", "mean"),
    avg_20d_volatility=("vol_avg_20", "mean"),
    avg_relative_volume=("rel_volume", "mean"),
    avg_amihud=("amihud", "mean"),
    avg_return_zscore=("zscore_ret_60", "mean")
)

regime_stats["frequency"] = (
    regime_stats["observations"] / len(train)
)

display(regime_stats.round(4))


## 12. Plot training regimes

In [ ]:

fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(train.index, train["Close"], linewidth=1, label="Close")

for state in range(N_STATES):
    mask = train["regime"] == state
    ax.scatter(
        train.index[mask],
        train.loc[mask, "Close"],
        s=3,
        label=f"State {state}"
    )

ax.set_title("Training Period — HMM Regimes")
ax.set_xlabel("Date")
ax.set_ylabel("Price")
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## 13. Out-of-sample regime inference on the final 5 years

The model and scaler are **frozen**. We do not refit them using test data.

In [ ]:

test["regime"] = hmm.predict(X_test)

test_posteriors = hmm.predict_proba(X_test)

for i in range(N_STATES):
    test[f"state_{i}_prob"] = test_posteriors[:, i]

display(test[["Close", "regime"]].head(10))


## 14. Out-of-sample regime characteristics

In [ ]:

test_regime_stats = test.groupby("regime").agg(
    observations=("Close", "size"),
    avg_daily_return=("logret_1d", "mean"),
    daily_volatility=("logret_1d", "std"),
    avg_5d_return=("ret_5d", "mean"),
    avg_20d_volatility=("vol_avg_20", "mean"),
    avg_relative_volume=("rel_volume", "mean"),
    avg_amihud=("amihud", "mean")
)

test_regime_stats["frequency"] = (
    test_regime_stats["observations"] / len(test)
)

display(test_regime_stats.round(4))


In [ ]:

fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(test.index, test["Close"], linewidth=1, label="Close")

for state in range(N_STATES):
    mask = test["regime"] == state
    ax.scatter(
        test.index[mask],
        test.loc[mask, "Close"],
        s=5,
        label=f"State {state}"
    )

ax.set_title("Out-of-Sample Test Period — HMM Regimes")
ax.set_xlabel("Date")
ax.set_ylabel("Price")
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## 15. Test whether regimes contain information about future returns

Because the regime itself is latent, there is no obvious 'correct label'. A useful test is whether the identified regimes have different forward-return distributions.

In [ ]:

FORWARD_HORIZONS = [1, 5, 10, 20, 60]

for h in FORWARD_HORIZONS:
    test[f"future_ret_{h}d"] = test["Close"].shift(-h) / test["Close"] - 1

forward_tables = {}

for h in FORWARD_HORIZONS:
    table = test.groupby("regime")[f"future_ret_{h}d"].agg(
        ["mean", "median", "std", "count"]
    )
    forward_tables[h] = table

    print(f"\n===== Forward {h}-day returns =====")
    display((table * [100, 100, 100, 1]).round(3))


## 16. Walk-forward next-day regime probabilities

This section is closer to genuine forecasting. For each test day, the model is updated using observations available **up to that day**, and the transition matrix is used to estimate the next day's regime probabilities.

The implementation below calculates the posterior probability of the current state using the data observed up to that point, rather than using the full future test sequence.

In [ ]:

def filtered_state_probabilities(model, X):
    """
    Forward-filtered state probabilities.

    alpha_t(i) = P(S_t=i | X_1,...,X_t)

    Returns one row per observation.
    """
    log_start = np.log(model.startprob_ + 1e-300)
    log_trans = np.log(model.transmat_ + 1e-300)

    # Gaussian log-likelihood for each observation/state.
    log_emission = model._compute_log_likelihood(X)

    log_alpha = np.empty_like(log_emission)

    log_alpha[0] = log_start + log_emission[0]

    # Stable log-sum-exp without requiring scipy explicitly.
    for t in range(1, len(X)):
        previous = log_alpha[t - 1][:, None] + log_trans
        max_previous = np.max(previous, axis=0)
        log_sum = max_previous + np.log(
            np.sum(np.exp(previous - max_previous), axis=0)
        )
        log_alpha[t] = log_emission[t] + log_sum

    max_alpha = np.max(log_alpha, axis=1, keepdims=True)
    probs = np.exp(log_alpha - max_alpha)
    probs /= probs.sum(axis=1, keepdims=True)

    return probs


# Filter the training data first.
# This gives the state distribution at the end of the training sample.
train_filtered = filtered_state_probabilities(hmm, X_train)
last_train_prob = train_filtered[-1]

# Sequentially filter the test data.
test_filtered = filtered_state_probabilities(hmm, X_test)

# Current filtered state
test["filtered_regime"] = np.argmax(test_filtered, axis=1)

# Probability of each current state
for i in range(N_STATES):
    test[f"filtered_state_{i}_prob"] = test_filtered[:, i]

# Next-day state probabilities:
# P(S_{t+1} | X_1,...,X_t) = P(S_t | X_1,...,X_t) * transition matrix
next_day_probs = test_filtered @ hmm.transmat_

for i in range(N_STATES):
    test[f"next_day_state_{i}_prob"] = next_day_probs[:, i]

test["predicted_next_day_regime"] = np.argmax(next_day_probs, axis=1)

display(
    test[
        ["Close", "filtered_regime", "predicted_next_day_regime"] +
        [f"next_day_state_{i}_prob" for i in range(N_STATES)]
    ].head(15)
)


## 17. Evaluate next-day regime forecasting

We compare the predicted next-day regime with the filtered regime actually assigned to the following observation. This is not a ground-truth label, but it measures how persistent/predictable the HMM states are out of sample.

In [ ]:

test["actual_next_day_filtered_regime"] = test["filtered_regime"].shift(-1)

valid = test.dropna(subset=["actual_next_day_filtered_regime"]).copy()

accuracy = (
    valid["predicted_next_day_regime"].astype(int)
    == valid["actual_next_day_filtered_regime"].astype(int)
).mean()

print(f"Next-day regime persistence/prediction agreement: {accuracy:.2%}")


## 18. Regime duration in the test period

In [ ]:

def regime_runs(regime_series):
    s = regime_series.dropna().astype(int)
    group_id = s.ne(s.shift()).cumsum()

    runs = (
        s.groupby(group_id)
         .agg(
             regime="first",
             start=lambda x: x.index.min(),
             end=lambda x: x.index.max(),
             duration_days="size"
         )
         .reset_index(drop=True)
    )
    return runs

runs = regime_runs(test["filtered_regime"])

display(runs.head(20))

duration_summary = runs.groupby("regime")["duration_days"].agg(
    ["count", "mean", "median", "min", "max"]
)

display(duration_summary.round(2))


## 19. Save the results

The saved CSV contains the original OHLCV data, engineered features, detected regimes, and state probabilities.

In [ ]:

OUTPUT_PATH = "hmm_market_regime_test_results.csv"

output_cols = [
    "Open", "High", "Low", "Close", "Volume",
    *FEATURES,
    "regime",
    "filtered_regime",
    "predicted_next_day_regime",
]

output_cols += [
    f"state_{i}_prob" for i in range(N_STATES)
]

output_cols += [
    f"filtered_state_{i}_prob" for i in range(N_STATES)
]

output_cols += [
    f"next_day_state_{i}_prob" for i in range(N_STATES)
]

output = test[[c for c in output_cols if c in test.columns]].copy()
output.to_csv(OUTPUT_PATH)

print(f"Saved: {OUTPUT_PATH}")



## 20. Recommended next steps

Before using this model for a trading strategy, investigate:

1. **Number of states:** compare 2–6 states across multiple random seeds.
2. **Feature selection:** test smaller and larger feature sets; remove redundant highly correlated features.
3. **Walk-forward validation:** instead of one 15/5 split, use expanding-window or rolling-window tests.
4. **Regime stability:** check whether similar regimes appear when the training window changes.
5. **Forward returns:** test 1/5/10/20/60-day returns conditional on the filtered regime.
6. **Transaction costs:** include realistic costs and slippage before evaluating a strategy.
7. **Look-ahead bias:** use filtered probabilities for real-time decisions; do not use full-sample Viterbi labels as if they were known in real time.
8. **Feature timing:** ensure every feature available at time *t* is constructed only from information available at or before *t*.
9. **Market-specific issues:** account for splits, dividends, contract rolls, holidays, missing sessions, and changes in trading hours where relevant.

### Important distinction

- `hmm.predict(X)` → best for **historical regime segmentation**.
- Forward-filtered probabilities → better for **real-time regime monitoring / next-day forecasting**.
- A serious trading model should be evaluated with a **walk-forward out-of-sample process**.
